In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("loan_data.csv")

In [3]:
X = df.drop("loan_status", axis=1)

y = df["loan_status"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [5]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder

In [6]:
from sklearn.compose import ColumnTransformer

In [7]:
preprocessing = ColumnTransformer(
    transformers=[
        ('OneHotEncoder', OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']),
        ('OrdinalEncoder',OrdinalEncoder(categories=[['High School','Bachelor','Associate','Master','Doctorate']]),['person_education'])
    ]
)

In [8]:
from sklearn.pipeline import Pipeline

In [9]:
#from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [10]:
model = SVC()

In [11]:
main_pipeline = Pipeline([
    ("preprocessor", preprocessing),
    ("model", SVC(random_state=42))
])

In [12]:
paramgrid = {'model__C':[0.01,0.1,1.0,10,100],
            'model__kernel':['linear','poly','rbf','sigmoid']}

In [13]:
from sklearn.model_selection import GridSearchCV

In [14]:
grid = GridSearchCV(estimator=main_pipeline,param_grid=paramgrid, n_jobs=-1, verbose=2)

In [ ]:
grid.fit(X_train,y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


### GridSearchCv

In [ ]:
#### scenario 1 :

# gridseachcv = GridSearchCV(
#     estimator = DecisionTreeClassifier(),
#     param_grid = {"max_depth":[None,10,50,100]},
#     cv = 5
# )


# #### Scenario 2

# gridseachcv = GridSearchCV(
#     estimator = main_pipeline,
#     # param_grid = {"max_depth":[None,10,50,100]},
#     param_grid = {"model__max_depth":[None,10,50,100]}, --->pipeline doesnot have max_step but model has to access that we use model__max_depth
#     cv = 5
# )

## Scenario 1

In [ ]:
# cv = 5 ; folds=5                                train                    test

# fold 1 = 2 -----> 3                          f1,f2,f3,f4                 f5
# fold 2 = 2 -----> 3                          f1,f3,f4,f5                 f2
# fold 3 = 2 -----> 3                          f1,f2,f4,f5                 f3
# fold 4 = 2 -----> 3                          f1,f2,f3,f5                 f4
# fold 5 = 2 -----> 1                          f2,f3,f4,f5                 f1

# cv 1 = 5;  train 80%
#            test 20%

#                 5 folds            |---> cv1 : 90% ---|
# 1. None,gini --------------------->|---> cv2 : 80% ---| Mean
# 2. None,Entropy                    |---> cv3 : 75% ---|----------> 81% ------|
# 3. 10, gini                        |---> cv4 : 70% ---|                      |
# 4. 10,Entropy                      |---> cv5 : 90% ---|                      |
# 5. 50,gini                                  ...                              |--> Choose the best parameters combination
# 6. 50,Entropy                               ...                              |    Based on the highest score
# 7. 100,Gini        5 folds                  ...                              |
# 8. 100,Entropy ------------------->         ...  ----------------> 95 % -----|


n_jobs = -1 / 1 / 2 / 3 / 4
          |-----> use all cpu cores